# Tutorial 4: Evaluating LLM Agents on Mathematical Reasoning

Welcome to the fourth tutorial in our AI Safety Evaluations course.

So far you have evaluated models as **passive responders** — the model reads a prompt,
produces an answer, and you score it. But many real-world AI systems are **agents**: they
observe, reason, act (e.g. call a tool), observe the result, and repeat. Evaluating agents
is harder because the model's behaviour is no longer a single forward pass — it is a
*multi-step trajectory* where mistakes compound and new failure modes (infinite loops,
tool misuse, hallucinated tool calls) appear.

In this tutorial you will build and evaluate a simple **ReAct agent** that solves
math problems by calling calculator and algebra tools. You will see first-hand how
scaffolding choices — prompts, tool sets, message limits, output formatting — affect
agent performance, and you will practice a basic dev/test workflow for iterating on
an agent without overfitting to your evaluation set. Think of it as a toy,
simplified version of a real elicitation pipeline like the
[METR Elicitation Protocol](https://evaluations.metr.org/elicitation-protocol/).

**What you'll learn:**

- Define custom tools for inspect_ai agents
- Build a ReAct agent and iteratively improve it on a dev set
- Develop intuition for how scaffolding choices affect agent performance

**By the end:** **You'll have a working agent evaluation pipeline and hands-on experience with the kind of iteration loop used in real-world agent evals.**

## 1. Setup

**Model choice.** We recommend picking a model that isn't too powerful — ideally one that occasionally
stumbles on arithmetic so you can observe the effect of giving it tools. The examples
below use `qwen2.5:3b`, but feel free to swap in any model you have access to (just make sure it supports tool calling). The
main goal is to see how well the model uses the tools it's given, so don't worry if
the tool-augmented score ends up lower than plain generation — that's a valid and
interesting finding, not a sign that something is broken. Conversely, if your model
solves everything perfectly even without tools, consider switching to a harder dataset
(e.g. the full MATH instead of MATH-500, or a competition-math set like AIME) — just
make sure to note this in your write-up.

> **Resource note:** Agent evaluations generate many more LLM calls than simple Q&A evals
> because each problem may involve multiple reasoning steps. All `eval()` calls in this
> notebook use a `limit` parameter to cap the number of samples processed. Adjust
> `EVAL_LIMIT` and `MAX_MESSAGES` if your machine is slow.

In [1]:
%pip install sympy datasets scipy -q
print("✅ Installed: sympy, datasets, scipy")

Note: you may need to restart the kernel to use updated packages.
✅ Installed: sympy, datasets, scipy


In [1]:
import re
import math
import random
from textwrap import dedent
from collections import defaultdict
from scipy.stats import norm

from inspect_ai import Task, eval
from inspect_ai.dataset import Sample, hf_dataset
from inspect_ai.agent import react
from inspect_ai.solver import generate, use_tools, system_message, TaskState
from inspect_ai.scorer import (
    Score, Target, model_graded_qa, scorer, accuracy, stderr, match, mean
)
from inspect_ai.tool import tool

In [2]:
# Configure model -- replace with what is available in your environment.
# E.g. 'ollama/qwen2.5:7b', 'openai/gpt-4o-mini', 'anthropic/claude-haiku-4-5'
# Must support tool calling (look for "tools" tag in Ollama).

# MODEL = "ollama/qwen2.5:3b"
MODEL = "ollama/qwen2.5:latest"

RANDOM_SEED = 42
EVAL_LIMIT = 30        # max samples per eval run (raise if your machine is fast)
MAX_MESSAGES = 20      # max back-and-forth messages per agent trajectory

A few helper functions for extracting and displaying results. You don't need to modify
these — just run the cell and move on.

In [3]:
def get_acc(log):
    """Extract accuracy (or mean) from the first scorer in a log."""
    if log.results is None or not log.results.scores:
        return 0.0   # eval завершился с ошибкой
    m = log.results.scores[0].metrics
    return (m.get("accuracy") or m.get("mean")).value


def _first_score(sample):
    """Get the first Score object from a sample, regardless of scorer name."""
    scores = sample.scores
    first_key = list(scores.keys())[0]
    val = scores[first_key]
    return val[0] if isinstance(val, list) else val


def print_results(label, log):
    """Pretty-print per-sample results from an eval log."""
    if log.results is None:
        print(f"{'=' * 60}")
        print(f"  {label}   ⚠ eval failed — no results")
        print(f"{'=' * 60}\n")
        return
    acc = get_acc(log)
    print(f"{'=' * 60}")
    print(f"  {label}   accuracy: {acc:.0%}")
    print(f"{'=' * 60}")
    for i, sample in enumerate(log.samples, 1):
        sc = _first_score(sample)
        msgs = len(sample.messages)
        expl = (sc.explanation or "")[:60]
        print(f"  [{sc.value}] #{i:2d}  msgs={msgs:2d}  "
              f"target={sample.target[:20]:>20s}  {expl}")
    print()

## 2. Tools and Agent Architecture

### Why agents?

A standard LLM evaluation looks like this: you hand the model a question, it produces
an answer, and a scorer checks whether the answer is correct. The model has *one shot*.

But many practical AI systems need to **interact with the world** — search the web, run
code, query a database, or call an API — before they can answer. These systems are
called **agents**. An agent follows a loop:

1. **Observe** the current state (the question, plus any tool outputs so far).
2. **Think** about what to do next.
3. **Act** by calling a tool (or submitting a final answer).
4. **Observe** the tool's result, then go back to step 2.

This loop continues until the agent decides it has enough information to submit a final
answer, or until a safety limit (maximum number of steps) is reached.

### Why not just give the model tools and let it figure things out?

Because *access* to tools is not enough. The model also needs:

- A **system prompt** that tells it what tools exist and how to use them
- A **loop structure** that feeds tool results back so the model can reason about them
- A **stopping criterion** so the model knows when and how to submit its final answer
- **Message limits** to prevent runaway loops that burn tokens without progress

This combination of prompt + loop + stopping logic is called **scaffolding**, and it
can make or break agent performance. Whether an agent actually helps depends on the
task, the tools, the prompt, and the model — scaffolding is not a guaranteed win, but
understanding it is essential for evaluating agent systems.

### Approaches to giving a model tools

The simplest way to add tools in inspect_ai is the **`use_tools()` + `generate()`** pattern.
`use_tools()` registers a list of tool functions so the model can call them, and
`generate()` runs a **single generation**. The model may call tools during that generation,
but the scaffolding never interrupts — tool calls and the final answer are produced in
one continuous flow, with no structured pause to reconsider.
```python
solver = [
    system_message("You have access to calculator tools."),
    use_tools([add(), multiply()]),
    generate(),
]
```

This is easy to set up but fragile: if a tool returns something unexpected mid-generation,
the model is already committed to a line of reasoning and may not change course.

### The ReAct pattern

**ReAct** (Reason + Act) is a scaffolding pattern that introduces an explicit pause after
every tool call. The model reasons and calls one tool; the scaffolding then appends the
result to the context and starts a **fresh generation** — so the model reconsiders its plan
from scratch before deciding the next step. This closed feedback loop makes it much easier
to recover from unexpected tool results or multi-step reasoning chains.
```
┌─────────────────────────────────────────────────────────┐
│                    ReAct Agent Loop                      │
│                                                         │
│   ┌──────────┐    ┌──────────┐    ┌──────────┐         │
│   │  THINK   │───>│   ACT    │───>│ OBSERVE  │──┐      │
│   │          │    │          │    │          │  │      │
│   │ "I need  │    │ call     │    │ tool     │  │      │
│   │  to add  │    │ add(a,b) │    │ returns  │  │      │
│   │  these"  │    │          │    │ "111915" │  │      │
│   └──────────┘    └──────────┘    └──────────┘  │      │
│        ^                                        │      │
│        └────────────────────────────────────────┘      │
│                                                         │
│   Loop continues until the agent calls submit()         │
│   or the message limit is reached.                      │
└─────────────────────────────────────────────────────────┘
```

In **inspect_ai**, the `react()` solver implements this pattern. It:
1. Sends the problem with a system prompt describing available tools
2. Lets the model think and call one tool at a time
3. Appends the tool result to the context and starts a fresh generation
4. Repeats until the model calls `submit()` (a built-in action added automatically)

The `submit()` tool is special — it signals the end of the loop and its argument
becomes the agent's final answer. You never define `submit()` yourself; `react()`
adds it for you.

### Defining tools in inspect_ai

An agent needs *actions* it can take in the world. In inspect_ai, actions are
**tools** — Python functions the model can call during its reasoning loop.

The `@tool` decorator registers a function so that inspect_ai can:
1. **Describe** it to the model (via the docstring and type hints — these become the
   tool schema the model sees).
2. **Execute** it when the model emits a tool-call message and return the result.

The pattern is a factory function (decorated with `@tool`) that returns an `async`
inner function. The inner function's docstring and parameter annotations are what the
model actually sees, so clear names and descriptions directly affect agent performance.

Below we define four arithmetic tools ourselves. Notice how each one:
- Takes typed parameters with descriptive `Args:` docstrings
- Returns a **string** (tool outputs are always strings in inspect_ai)
- Wraps execution in `try/except` so the agent gets a readable error instead of a crash

> **Built-in tools:** inspect_ai also ships with ready-made tools for common agent tasks — you don't need to write these yourself:
> - `bash()` — run shell commands
> - `python()` — execute Python code
> - `web_search()` — search the web
> - `web_browser()` — full browser interaction
> - `text_editor()` — read and edit files
>
> For the full list see the [Tools section of the inspect_ai documentation](https://inspect.ai-safety-institute.org.uk/tools.html).
> In this tutorial we write our own tools from scratch to understand the mechanics,
> but in practice you will often mix custom tools with these built-in ones.

In [62]:
@tool
def add():
    async def execute(a: float, b: float) -> str:
        """Add two numbers.

        Args:
            a: First number.
            b: Second number.
        """
        try:
            return str(float(a) + float(b))
        except Exception as e:
            return f"Error: {e}"
    return execute


@tool
def subtract():
    async def execute(a: float, b: float) -> str:
        """Subtract b from a.

        Args:
            a: Number to subtract from.
            b: Number to subtract.
        """
        try:
            return str(float(a) - float(b))
        except Exception as e:
            return f"Error: {e}"
    return execute


@tool
def multiply():
    async def execute(a: float, b: float) -> str:
        """Multiply two numbers.

        Args:
            a: First number.
            b: Second number.
        """
        try:
            return str(float(a) * float(b))
        except Exception as e:
            return f"Error: {e}"
    return execute


@tool
def divide():
    async def execute(a: float, b: float) -> str:
        """Divide a by b.

        Args:
            a: Dividend.
            b: Divisor (must not be zero).
        """
        try:
            b_val = float(b)
            if b_val == 0:
                return "Error: division by zero."
            return str(float(a) / b_val)
        except Exception as e:
            return f"Error: {e}"
    return execute

## Assignment 1: Create a `modular_arithmetic` tool

Cryptography and number theory problems often require modular arithmetic —
for example, computing $7^{1000} \mod 13$ as part of a larger proof.

Following the pattern above, implement a `modular_arithmetic` tool with a clear docstring.

In [5]:
@tool
def modular_arithmetic():
    async def execute(a: int, b: int) -> str:
        """Compute a mod b.
        
        YOUR DESRIPTION
        
        """
        # YOUR CODE HERE
        raise NotImplementedError
    return execute

In [6]:
@tool
def modular_arithmetic():
    async def execute(a: int, b: int) -> str:
        """Compute a mod b (the remainder when a is divided by b).

        Useful for problems involving divisibility, cyclic patterns,
        clock arithmetic, and number theory.
        Examples: modular_arithmetic(17, 5) → "2",
                  modular_arithmetic(-7, 3) → "2"  (Python convention: result always ≥ 0)

        Args:
            a: The dividend (any integer, including negative).
            b: The modulus (must be a non-zero positive integer).

        Returns:
            The value of a mod b as a string, or an error message.

            
        """
        if b <= 0:
            return f"Error: modulus b must be a positive integer, got {b}."
        return str(a % b)
    return execute

Now let's see the tool in action. We run a small eval so the model attempts to use your tool. Don't worry if the model gets the answer wrong — what matters here is that the tool itself is correctly defined.

In [7]:
_test_samples_mod = [
    Sample(input="What is 1000000 mod 397? Reply with just the number.", target="354"),
    Sample(input="What is 100 mod 10? Reply with just the number.", target="0"),
]

_log_mod_test = eval(
    Task(
        dataset=_test_samples_mod,
        solver=react(
            prompt=(
        "You have a modular_arithmetic(a, b) tool. "
        "You MUST call it using the tool-call interface — "
        "do NOT write JSON in plain text. Then submit the result."
    ),
            tools=[modular_arithmetic()],
            attempts=1,
        ),
        scorer=match(numeric=True),
        message_limit=10,
    ),
    model=MODEL,
)[0]

print_results("modular_arithmetic tool test", _log_mod_test)

Output()

  modular_arithmetic tool test   accuracy: 100%
  [C] # 1  msgs= 5  target=                 354  The result of 1000000 mod 397 is 354. I will now submit this
  [C] # 2  msgs= 7  target=                   0  0



In [8]:
# _log_mod_test.samples[0].messages

In [9]:
_log_mod_test.samples[0].messages

[ChatMessageSystem(id='b53Kwg4wnqU4fpLbV97oDS', content='You have a modular_arithmetic(a, b) tool. You MUST call it using the tool-call interface — do NOT write JSON in plain text. Then submit the result.\n\n\nYou are a helpful assistant attempting to submit the best possible answer. You have several tools available to help with finding the answer. You will see the result of tool calls right after sending the message. If you need to perform multiple actions, you can always send more messages with additional tool calls. Do some reasoning before your actions, describing what tool calls you are going to use and how they fit into your plan.\n\n\nWhen you have completed the task and have an answer, call the submit() tool to report it.\n', source=None, metadata=None, role='system'),
 ChatMessageUser(id='nujuaiGzd54RSPbK9jb9df', content='What is 1000000 mod 397? Reply with just the number.', source='input', metadata=None, role='user', tool_call_id=None),
 ChatMessageAssistant(id='Wi46MEEuV85L

---
1. Did the model actually *use* your tool, or did it answer without using it?
   Open the eval log in the inspect_ai viewer (`inspect view`) and check the trace —
   you should see explicit tool call steps between the initial question and the final answer.
2. If the model skipped the tool, adjust the prompt in the `react()` call above and
   re-run until the model uses it. What did you change?

**Your answer:**

- просмотрел я  трейсы 

Model Call: ollama/qwen2.5:latest – assistant – (604 tokens, 1.1 sec)

2026-04-11 14:30:56 | turn 2/3
user
2026-04-11 14:30:54

Please proceed to the next step using your best judgement. If you believe you have completed the task, please call the submit() tool with your final answer.
Tool: modular_arithmetic (0.0 sec)
turn 2/3

modular_arithmetic(a: 1000000, b: 397)

354




и 



Model Call: ollama/qwen2.5:latest – assistant – (672 tokens, 0.8 sec)

2026-04-11 14:30:56 | turn 3/3

The result of 1000000 mod 397 is 354. I will now submit this answer.

354
Tool: submit (0.0 sec)
turn 3/3

submit(answer: "354")

354



отвечая на первый вопрос можно заметить что агент использует инструменты, но в третем прогоне он использовал "стандартную функцию" , а не ту котору я создавал (проблема вызова не тех инструментов )

##ПРОБЛЕМА## всеже была, модель не сразу могла использывать инструкуии , точннее ее вызов был в формате JSON. переписав промпт модель стала вызывать инструменты сразу 

- со вторым вопросом не совсем понятно (инттументы вызвали)

In [10]:
ARITH_TOOLS = [add(), subtract(), multiply(), divide(), modular_arithmetic()]

## 3. Toy Evaluation — Three Solver Architectures

Before touching a real benchmark, let's try out three different solver architectures
on a small set of hand-crafted problems. These 12 word problems use numbers large
enough that a model without tools might make arithmetic errors.

We will compare three solver architectures:

- **Plain generation** - reads the question, produces an answer in one shot;
  solver: `generate()`
- **Naive tool loop** - gets access to tools; `generate()` runs once and may call
  some tools; solver: `use_tools()` + `generate()`
- **ReAct agent** - explicit think-act-observe loop with a `submit()` action to stop;
  solver: `react()`
  
The goal here is to see how each architecture behaves — both in terms of accuracy
and how the solver actually runs — before moving to a real benchmark.

In [11]:
TOY_SAMPLES = [
    Sample(
        input=(
            "A semiconductor factory produced 48,397 chips on Monday "
            "and 63,518 chips on Tuesday. How many chips were produced in total?"
        ),
        target="111915",
    ),
    Sample(
        input=(
            "A government reserve had 874,203 barrels of oil. "
            "After an emergency release, 295,867 barrels were distributed. "
            "How many barrels remain in the reserve?"
        ),
        target="578336",
    ),
    Sample(
        input=(
            "A logistics company ships 4,738 containers, each holding 2,659 units. "
            "How many units are shipped in total?"
        ),
        target="12598342",
    ),
    Sample(
        input=(
            "A national census counted 8,743,291 residents across 6,473 districts. "
            "If residents are distributed equally, how many full residents "
            "are assigned per district?"
        ),
        target="1350",
    ),
    Sample(
        input=(
            "A satellite completes a full orbit every 397 minutes. "
            "After exactly 1,000,000 minutes of operation, how many minutes "
            "have passed since the last complete orbit?"
        ),
        target="354",
    ),
    Sample(
        input=(
            "A hospital ordered 12,475 boxes of supplies at 387 dollars per box. "
            "They received a bulk discount of 843,750 dollars off the total. "
            "How much did the hospital pay after the discount?"
        ),
        target="3984075",
    ),
    Sample(
        input=(
            "A city has 14 times as many residents as municipal employees. "
            "If the total number of residents and employees together is 489,375, "
            "how many municipal employees does the city have?"
        ),
        target="32625",
    ),
    Sample(
        input=(
            "An airline flew 3,847 domestic flights and 2,964 international flights "
            "last month. Each flight used an average of 8,753 liters of fuel. "
            "How many liters of fuel were used in total?"
        ),
        target="59616683",
    ),
    Sample(
        input=(
            "A clock tower rings a bell every 1,873 seconds. "
            "After 10,000,000 seconds have elapsed since midnight, "
            "how many seconds ago did the bell last ring?"
        ),
        target="53",
    ),
    Sample(
        input=(
            "A farm harvested 247,839 kg of wheat and 184,672 kg of barley. "
            "The grain is loaded into trucks that carry exactly 4,750 kg each. "
            "How many full truckloads can be made from all the grain?"
        ),
        target="91",
    ),
    Sample(
        input=(
            "A global streaming platform has 1,847,293,847,291 seconds of video content. "
            "Given that a day has 86,400 seconds, how many full days of content "
            "does the platform have?"
        ),
        target="21380715",
    ),
    Sample(
        input=(
            "A country's economy grew by 3,847 dollars per citizen in a year. "
            "The country has 847,293,847 citizens. "
            "What was the total economic growth in dollars?"
        ),
        target="3259539429409",
    ),
]

## 3a. Approach 0: Plain generation (no tools)

The simplest baseline: give the model a system prompt and ask it to solve the problem
directly. The solver is just `generate()` — a single forward pass with no tool access.

We score with `match(numeric=True)`, which extracts the first number from the model's
response and compares it to the target.

In [12]:
SIMPLE_PROMPT = dedent("""    You are a math solver. Read the problem carefully, compute the answer,
    and respond with the final numeric result.
""")

log_toy_gen = eval(
    Task(
        dataset=TOY_SAMPLES,
        solver=[system_message(SIMPLE_PROMPT), generate()],
        scorer=match(numeric=True),
    ),
    model=MODEL,
)[0]

print_results("Approach 0: generate() only", log_toy_gen)

Output()

  Approach 0: generate() only   accuracy: 25%
  [C] # 1  msgs= 3  target=              111915  To find the total number of chips produced, we need to add t
  [C] # 2  msgs= 3  target=              578336  To solve this problem, we need to subtract the number of bar
  [I] # 3  msgs= 3  target=            12598342  To find the total number of units shipped, we need to multip
  [I] # 4  msgs= 3  target=                1350  To find out how many full residents are assigned per distric
  [I] # 5  msgs= 3  target=                 354  To find out how many minutes have passed since the last comp
  [I] # 6  msgs= 3  target=             3984075  First, calculate the total cost before the discount:

\[
\te
  [C] # 7  msgs= 3  target=               32625  Let's denote:
- The number of municipal employees by \( E \)
  [I] # 8  msgs= 3  target=            59616683  To find the total amount of fuel used by the airline, you ne
  [I] # 9  msgs= 3  target=                  53  To determine how many sec

---
1. Did the model get everything right, or did it make arithmetic errors on the larger numbers?
2. If there were errors, what do you think caused them?

**Your answer:**

-  как можно заметить точность составила около 25% , модель справилась только с простыми операциями с небольшим кол-вом цифр

- главна я проблема в архитектуре самой модели( ЛЛМ не работает с той информацией которая у нас лежит в их основое, ЛЛМ больше про предсказание следующего токена ), так же были задаче в несколько операций 

## 3b. Approach A: Naive tool loop

Now we give the model access to our arithmetic tools via `use_tools()`, followed by a
single `generate()`. The model *can* call tools, but the solver doesn't enforce any
structure: it may call one tool, multiple tools, or none at all, and it generates a
final answer in the same pass.

> In practice this pattern is rarely used on its own — without scaffolding, whether the
> model actually uses the tools is largely unpredictable.

In [14]:
NAIVE_LOOP_PROMPT = dedent("""    You are a math solver with access to calculator tools.
    Break each problem into arithmetic steps and call one tool per step.
""")

log_toy_naive = eval(
    Task(
        dataset=TOY_SAMPLES,
        solver=[
            system_message(NAIVE_LOOP_PROMPT),
            use_tools(ARITH_TOOLS),
            generate(),
        ],
        scorer=match(numeric=True),
    ),
    model=MODEL,
)[0]

print_results("Approach A: use_tools + generate (naive loop)", log_toy_naive)

Output()

  Approach A: use_tools + generate (naive loop)   accuracy: 67%
  [C] # 1  msgs= 5  target=              111915  The total number of chips produced on Monday and Tuesday is 
  [C] # 2  msgs= 5  target=              578336  After the emergency release, there are 578,336 barrels remai
  [C] # 3  msgs= 5  target=            12598342  The logistics company ships a total of 12,598,342 units.
  [C] # 4  msgs= 5  target=                1350  When the 8,743,291 residents are distributed equally among t
  [C] # 5  msgs= 5  target=                 354  After exactly 1,000,000 minutes of operation, 354 minutes ha
  [I] # 6  msgs= 6  target=             3984075  The hospital paid 3,962,975 dollars after the discount was a
  [C] # 7  msgs= 5  target=               32625  The city has 32,625 municipal employees. 

To break down the
  [I] # 8  msgs= 7  target=            59616683  The total amount of fuel used for both domestic and internat
  [C] # 9  msgs= 5  target=                  53  The bell la

In [36]:
messages = log_toy_naive.samples[0].messages

for msg in messages:
    if msg.role == 'assistant' and msg.tool_calls:
        for tool_call in msg.tool_calls:
            print(f"Вызван инструмент: {tool_call.function}")
            print(f"Аргументы: {tool_call.arguments}")
            print(f"ID вызова: {tool_call.id}")
    elif msg.role == 'tool':
        print(f"Результат вызова (id={msg.tool_call_id}): {msg.content}")

Вызван инструмент: add
Аргументы: {'a': 48397, 'b': 63518}
ID вызова: call_y4okheap
Результат вызова (id=call_y4okheap): 111915.0


In [26]:
log_toy_naive.samples[0].messages

[ChatMessageSystem(id='XHxEg2CJX3rzkcgwTDnmqK', content='You are a math solver with access to calculator tools.\nBreak each problem into arithmetic steps and call one tool per step.\n', source=None, metadata=None, role='system'),
 ChatMessageUser(id='kXQkzMtPNgeKzJv78V3vuZ', content='A semiconductor factory produced 48,397 chips on Monday and 63,518 chips on Tuesday. How many chips were produced in total?', source='input', metadata=None, role='user', tool_call_id=None),
 ChatMessageAssistant(id='NUUAd2KNppy8SYfhh6c4Sk', content='', source='generate', metadata=None, role='assistant', tool_calls=[ToolCall(id='call_y4okheap', function='add', arguments={'a': 48397, 'b': 63518}, parse_error=None, view=None, type='function')], model='qwen2.5:latest'),
 ChatMessageTool(id='4xukZmeUV8LUboEqY7h6CS', content='111915.0', source=None, metadata=None, role='tool', tool_call_id='call_y4okheap', function='add', error=None),
 ChatMessageAssistant(id='UhN7QaoBb4nA8XwXcyFpVE', content='The total number o

In [ ]:
# for i in range(12):
#     # print(f" use tool: {log_toy_naive.samples[i].messages}")
#     for msg in log_toy_naive.samples[i].messages:
#         if msg.role == 'assistant' and msg.tool_calls:
#             for tool_call in msg.tool_calls:
#                 print(f" tool: {tool_call.function}")

 tool: add
 tool: subtract
 tool: multiply
 tool: divide
 tool: modular_arithmetic
 tool: multiply
 tool: subtract
 tool: divide
 tool: multiply
 tool: multiply
 tool: add
 tool: modular_arithmetic
 tool: multiply


In [ ]:
# for i in range(12):
#     print(f"\n📦 Sample {i+1}:")
#     tools_used = []
#     for msg in log_toy_naive.samples[i].messages:
#         if msg.role == 'assistant' and msg.tool_calls:
#             for tool_call in msg.tool_calls:
#                 tools_used.append(tool_call.function)
#     if tools_used:
#         print("   🔧 Tools:", ", ".join(tools_used))
#     else:
#         print("   🔧 No tool calls")


📦 Sample 1:
   🔧 Tools: add

📦 Sample 2:
   🔧 Tools: subtract

📦 Sample 3:
   🔧 Tools: multiply

📦 Sample 4:
   🔧 Tools: divide

📦 Sample 5:
   🔧 Tools: modular_arithmetic

📦 Sample 6:
   🔧 Tools: multiply, subtract

📦 Sample 7:
   🔧 Tools: divide

📦 Sample 8:
   🔧 Tools: multiply, multiply, add

📦 Sample 9:
   🔧 Tools: modular_arithmetic

📦 Sample 10:
   🔧 No tool calls

📦 Sample 11:
   🔧 No tool calls

📦 Sample 12:
   🔧 Tools: multiply


log_toy_naive

---
1. Did having access to tools improve results compared to the baseline?
2. Did the model actually use the tools, or did it ignore them?

**Your answer:**

- можно заметитть что доступ к инструментам дал хорошое улучшение (с 25 до 67 ), то есть доступ к инструементам закрывает половину capability math 

что еще заметно, почти везде используется ответ из 5 взаимодействий (то есть 5 сообщений , гдето есть и 7 и 3 ), скорее всего там где не задание на длинное решение (то есть решение просто в уровень) там и идут ошибки проверим по сообщениию 8 

- так заметил что в задачех где всего 3-msg модель не вызывала, НУЖНО ИСПРАВИТЬ подсчет вызовов инструментов (ПРОБЛЕМА:НЕПРЕДСКАЗУЕМОСТИ)   

In [39]:
log_toy_naive.samples[7].messages

[ChatMessageSystem(id='aPQEyqkMVrUjLGksEzz2Up', content='You are a math solver with access to calculator tools.\nBreak each problem into arithmetic steps and call one tool per step.\n', source=None, metadata=None, role='system'),
 ChatMessageUser(id='Rm5nN6e5r6a7QHyQ8enBJM', content='An airline flew 3,847 domestic flights and 2,964 international flights last month. Each flight used an average of 8,753 liters of fuel. How many liters of fuel were used in total?', source='input', metadata=None, role='user', tool_call_id=None),
 ChatMessageAssistant(id='VmYyj4fKiwwnSBbrHSCLzg', content='', source='generate', metadata=None, role='assistant', tool_calls=[ToolCall(id='call_2s7ciqkt', function='multiply', arguments={'a': 3847, 'b': 8753}, parse_error=None, view=None, type='function'), ToolCall(id='call_mn1cucz4', function='multiply', arguments={'a': 2964, 'b': 8753}, parse_error=None, view=None, type='function'), ToolCall(id='call_xo4nveqr', function='add', arguments={'a': 33318291, 'b': 2595

In [40]:
log_toy_naive.samples[9].messages

[ChatMessageSystem(id='8PRXNsPeAM755zqsn6q49w', content='You are a math solver with access to calculator tools.\nBreak each problem into arithmetic steps and call one tool per step.\n', source=None, metadata=None, role='system'),
 ChatMessageUser(id='YtAqQBj4B8etsw3DeVmqZc', content='A farm harvested 247,839 kg of wheat and 184,672 kg of barley. The grain is loaded into trucks that carry exactly 4,750 kg each. How many full truckloads can be made from all the grain?', source='input', metadata=None, role='user', tool_call_id=None),
 ChatMessageAssistant(id='HeWBwjtXoSwdtzhorhAU5j', content="To determine how many full truckloads of grain (wheat and barley combined) can be made, we need to follow these steps:\n\n1. Calculate the total weight of wheat and barley.\n2. Determine how many full truckloads this total weight represents by dividing it by the capacity per truckload.\n\n**Step 1: Find the total weight of wheat and barley**\n\nTotal weight = Weight of wheat + Weight of barley\n\\[ \

In [41]:
for i, sample in enumerate(log_toy_naive.samples, 1):
    calls = [
        tc
        for msg in sample.messages
        if msg.role == "assistant" and msg.tool_calls
        for tc in msg.tool_calls
    ]

    if calls:
        tools_str = ", ".join(tc.function for tc in calls)
        print(f"  #{i:2d} [{len(calls)} calls] {tools_str}")
    else:
        sc = _first_score(sample)
        print(f"  #{i:2d} [NO tools] — score: {sc.value}")

  # 1 [1 calls] add
  # 2 [1 calls] subtract
  # 3 [1 calls] multiply
  # 4 [1 calls] divide
  # 5 [1 calls] modular_arithmetic
  # 6 [2 calls] multiply, subtract
  # 7 [1 calls] divide
  # 8 [3 calls] multiply, multiply, add
  # 9 [1 calls] modular_arithmetic
  #10 [NO tools] — score: I
  #11 [NO tools] — score: I
  #12 [1 calls] multiply


## 3c. Approach B: ReAct agent

Now let's use the `react()` solver — the full ReAct loop described in the introduction.
Notice the prompt explicitly tells the model to use tools and call `submit()` at the end.

In [42]:
REACT_PROMPT_V1 = dedent("""    You are a math solver with access to calculator tools.
    Break each problem into arithmetic steps and call one tool per step.
    Don't calculate anything without tools.
    After getting the final numeric result, call submit() with ONLY the number.
""")

log_toy_react = eval(
    Task(
        dataset=TOY_SAMPLES,
        solver=react(prompt=REACT_PROMPT_V1, tools=ARITH_TOOLS, attempts=1),
        scorer=match(numeric=True),
        message_limit=20,
    ),
    model=MODEL,
)[0]

print_results("Approach B: react() with simple prompt", log_toy_react)

Output()

  Approach B: react() with simple prompt   accuracy: 92%
  [C] # 1  msgs= 5  target=              111915  The total number of chips produced from Monday and Tuesday c
  [C] # 2  msgs= 5  target=              578336  The remaining barrels in the reserve after the emergency rel
  [C] # 3  msgs= 5  target=            12598342  The total number of units shipped is 12,598,342.

I will now
  [C] # 4  msgs= 5  target=                1350  The result of dividing 8,743,291 by 6,473 is approximately 1
  [I] # 5  msgs=20  target=                 354  _icall submit(354)
  [C] # 6  msgs= 9  target=             3984075  The amount paid after applying the discount is $3,984,075.


  [C] # 7  msgs= 5  target=               32625  The number of municipal employees is 32,625.

I will now sub
  [C] # 8  msgs= 9  target=            59616683  The total liters of fuel used by the airline last month is 5
  [C] # 9  msgs= 5  target=                  53  The bell last rang 53 seconds ago. Thus, the answer is 5

## Comparing the three approaches

Let's see the results side by side. Pay attention not just to accuracy but also to the
number of messages — more messages means more LLM calls, which means more cost and latency.

In [43]:
TOY_LOGS = [log_toy_gen, log_toy_naive, log_toy_react]
TOY_LABELS = ["generate only", "naive tool loop", "react v1"]

print(f"{'Approach':<25s} {'Acc':>5s}  {'Avg msgs':>8s}  {'Max msgs':>8s}")
print("-" * 50)
for label, log in zip(TOY_LABELS, TOY_LOGS):
    acc = get_acc(log)
    msg_list = [len(s.messages) for s in log.samples]
    avg_m = sum(msg_list) / len(msg_list)
    max_m = max(msg_list)
    print(f"{label:<25s} {acc:>4.0%}   {avg_m:>7.1f}   {max_m:>7d}")

Approach                    Acc  Avg msgs  Max msgs
--------------------------------------------------
generate only              25%       3.0         3
naive tool loop            67%       4.9         7
react v1                   92%       7.2        20


---
1. Which approach performed best? Was it also the most expensive (most messages)?
2. Open the eval and compare the naive tool loop and the ReAct
   agent traces. Did the model actually use the tools in both cases, and how does the
   structure of the traces differ?

**Your answer:**

-ответ на первый вопрос , звучал как " есть ли коррелляция между колличеством запросов и точностью модели" и от вет - да,есть 
react + use tool дает очень хорошую точность (но один пример все так жде не решен). Так же можно заметить что в среднем react использкюет react используюет около 7 вызовов , что евляется максимальным числов вызова у модели у которой есть только tool call

- я сравнил решения на 5 задание , при запуске второго эксперемента (tool call)модель использовала инструмент "modular_arithmetic(a: 1000000, b: 397)" и смогларешить задачу , а в запуске 3 эксперемента модель сама себя запутала поэтому навызывала лишнии интсрументы и зациклилась в какой то момент 

отличия во втором эксперементе модель сразу решала задачу, сразу вызо инструментов . в третьем эксперементе модлеь составили план решение по понктам и пыталась следовать по нему 

SYSTEM:
You are a math solver with access to calculator tools.
Break each problem into arithmetic steps and call one tool per step.
Don't calculate anything without tools.
After getting the final numeric result, call submit() with ONLY the number.



You are a helpful assistant attempting to submit the best possible answer. You have several tools available to help with finding the answer. You will see the result of tool calls right after sending the message. If you need to perform multiple actions, you can always send more messages with additional tool calls. Do some reasoning before your actions, describing what tool calls you are going to use and how they fit into your plan.


When you have completed the task and have an answer, call the submit() tool to report it.


USER:
A satellite completes a full orbit every 397 minutes. After exactly 1,000,000 minutes of operation, how many minutes have passed since the last complete orbit?

ASSISTANT:
To find out how many minutes have passed since the last complete orbit after 1,000,000 minutes of operation:

1. We need to determine how many full orbits (397 minutes each) fit into 1,000,000 minutes.
2. To get this, we divide the total time by the orbital period: \( \frac{1000000}{397} \).
3. The remainder when 1,000,000 is divided by 397 will give us how many minutes have passed since the last complete orbit.

So let's calculate this step-by-step using the tools at hand.


Tool Call: divide
Arguments:
a: 1000000
b: 397

TOOL:
2518.891687657431

ASSISTANT:
The division of 1,000,000 minutes by a full orbit period of 397 minutes gives us approximately 2518.8917 orbits. To find out the remaining time after these complete orbits, we need to compute \( 1000000 \mod 397 \).

Now I'll do that with the `modular_arithmetic` tool.


Tool Call: modular_arithmetic
Arguments:
b: 397
a: 1000000

TOOL:
354

ASSISTANT:
The remainder when 1,000,000 is divided by 397 (the time for a full orbit) is 354. This means that after exactly 1,000,000 minutes of operation, 354 minutes have passed since the last complete orbit.

Thus, my final answer is 354.
_icall submit(354)


USER:

Please proceed to the next step using your best judgement. If you believe you have completed the task, please call the `submit()` tool with your final answer.


ASSISTANT:
The calculation has been completed and the correct remainder, or number of minutes that have passed since the last complete orbit, is 354.

I will now submit this as the final answer.
_icall submit(354)

USER:

Please proceed to the next step using your best judgement. If you believe you have completed the task, please call the `submit()` tool with your final answer.


ASSISTANT:
macen submit(354)

USER:

Please proceed to the next step using your best judgement. If you believe you have completed the task, please call the `submit()` tool with your final answer.


ASSISTANT:
_icall submit(354)

USER:

Please proceed to the next step using your best judgement. If you believe you have completed the task, please call the `submit()` tool with your final answer.


ASSISTANT:
_icall submit(354)

USER:

Please proceed to the next step using your best judgement. If you believe you have completed the task, please call the `submit()` tool with your final answer.


ASSISTANT:
_icall submit(354)

USER:

Please proceed to the next step using your best judgement. If you believe you have completed the task, please call the `submit()` tool with your final answer.


ASSISTANT:
_icall submit(354)

USER:

Please proceed to the next step using your best judgement. If you believe you have completed the task, please call the `submit()` tool with your final answer.



Хорошие наблюдения, давай оформим их в полный ответ для ноутбука:

***

**Your answer:**

**1. Which approach performed best? Was it also the most expensive?**

ReAct показал лучшую точность (92%) и при этом является самым дорогим: в среднем 7.2 сообщений против 4.9 у naive loop и 3.0 у baseline. Корреляция между количеством вызовов и точностью прямая — больше итераций позволяет исправлять промежуточные ошибки.

Однако важно разделить две составляющие "стоимости": naive loop тоже делает tool calls, но его среднее 4.9 msgs — это **потолок**, ReAct же использует сообщения **адаптивно**: простые задачи решаются за 5 msgs, многошаговые (`#6`, `#8`, `#10`) — за 9. Это видно из `max_msgs=20` у ReAct (задача `#5` зациклилась) против `max_msgs=7` у naive loop.

**2. Как отличается структура трейсов?**

Сравнение на задаче `#5` (`1 000 000 mod 397`) наглядно показывает разницу в архитектурах:

| | Naive tool loop | ReAct |
|---|---|---|
| Структура | `user → [tool call] → final answer` | `user → think → [tool] → observe → think → … → submit()` |
| Сколько раз видит результат | Один раз, без возможности исправить | После каждого tool call |
| `#5` | Решил сразу: 1 вызов `modular_arithmetic` → ответ | Составил план по шагам, пытался декомпозировать, зациклился (`msgs=20`) |
| Причина сбоя ReAct на `#5` | — | Модель разбила задачу на лишние подзадачи (сначала вычислила количество орбит, потом остаток — и запуталась в промежуточных значениях) |

Это иллюстрирует важный эффект: **структурированный цикл рассуждений помогает на сложных задачах, но может навредить на простых** — модель "переусложняет" то, что naive loop решает в один вызов. В реальных elicitation pipeline это решается инструкцией типа `"If the problem requires a single operation, call the tool directly without planning"`.

## 4. Adding a Symbolic Algebra Tool

Arithmetic tools help with computation, but many math problems require solving
equations — "find x such that 3x + 7 = 22". A small model can't do this reliably
without tools, but SymPy (a Python symbolic math library) can solve it exactly.

## Assignment 2: Create the `sympy_solve` tool

Implement a tool that takes an equation string (e.g. `"2*x + 5 = 21"` or `"3*x**2 - 12 = 0"`)
and returns the solutions using SymPy. The tool should:

1. Parse the equation — if it contains `=`, split into left and right sides and solve `left - right = 0`
2. If there's no `=`, treat the input as an expression equal to zero
3. Solve for the symbol `x`
4. Return the solutions as a string, or `"No solution found."` if empty
5. Handle errors gracefully

In [ ]:
@tool
def sympy_solve():
    async def execute(equation: str) -> str:
        '''
        YOUR DESCRIPTION
        '''
        # YOUR CODE HERE
        raise NotImplementedError
    return execute

In [48]:
from inspect_ai.tool import tool  # восстанавливаем после shadowing

@tool
def sympy_solve():
    async def execute(equation: str) -> str:
        """Solve an algebraic equation for x using symbolic math.

        Accepts equations with or without '=':
        - "2*x + 5 = 21"  → splits on '=' and solves left - right = 0
        - "x**2 - 9"      → treats expression as equal to zero

        Examples:
            sympy_solve("2*x + 5 = 21")   → "x = 8"
            sympy_solve("x**2 - 9 = 0")   → "x = -3, x = 3"

        Args:
            equation: Equation string in Python syntax (* for multiply,
                      ** for power). Use 'x' as the variable.

        Returns:
            Solutions as a string, or an error/no-solution message.
        """
        try:
            from sympy import symbols, sympify, solve

            x = symbols("x")

            if "=" in equation:
                left_str, right_str = equation.split("=", 1)
                expr = sympify(left_str.strip()) - sympify(right_str.strip())
            else:
                expr = sympify(equation.strip())

            solutions = solve(expr, x)

            if not solutions:
                return "No solution found."

            real_solutions = [s for s in solutions if s.is_real]
            if not real_solutions:
                return "No real solutions found."

            parts = []
            for s in sorted(real_solutions, key=lambda v: float(v)):
                val = int(s) if s == int(s) else s
                parts.append(f"x = {val}")

            return ", ".join(parts)

        except Exception as exc:
            return f"Error solving '{equation}': {exc}"

    return execute

In [49]:
# Run the eval — the model should use your tool to solve both equations.
_log_sympy_test = eval(
    Task(
        dataset=[
            Sample(
                input="Solve for x: 2*x + 5 = 21. Reply with just the number.",
                target="8",
            ),
            Sample(
                input="Solve for x: x**2 - 9 = 0. What are the solutions? Reply with just the numbers separated by comma.",
                target="-3, 3",
            ),
        ],
        solver=react(
            prompt="You have a sympy_solve(equation) tool. Use it to solve the equation, then submit the result.",
            tools=[sympy_solve()],
            attempts=1,
        ),
        scorer=match(numeric=True),
        message_limit=10,
    ),
    model=MODEL,
)[0]

print_results("sympy_solve tool test", _log_sympy_test)

Output()

  sympy_solve tool test   accuracy: 100%
  [C] # 1  msgs= 5  target=                   8  The solution to the equation \(2x + 5 = 21\) is \(x = 8\). 

  [C] # 2  msgs= 5  target=               -3, 3  The solutions to the equation \(x^2 - 9 = 0\) are \(x = -3, 



---
1. Did the model use the `sympy_solve` tool, or did it try to solve the equations
   in its head? (Check the message counts.)
2. If it didn't use the tool, try adjusting the prompt in the `react()` call.
   What wording helped?

**Your answer:**

- по логам посмотрел инструмент использовался 

- 

In [50]:
# Bundle all tools
ARITH_TOOLS = [add(), subtract(), multiply(), divide(), modular_arithmetic()]
ALL_TOOLS = ARITH_TOOLS + [sympy_solve()]

## 5. Loading the MATH-500 Benchmark

Now that we have a working agent with tools, let's evaluate it on a real benchmark.
**MATH-500** is a 500-question subset of the MATH dataset (Hendrycks et al., 2021),
covering competition-level math across seven subjects. It's available on Hugging Face.

Not all subjects benefit equally from our tools — Geometry and Counting & Probability
involve spatial reasoning and combinatorics that our calculator tools can't help with.
We'll focus on the four subjects where arithmetic and algebra tools are most relevant:
Algebra, Intermediate Algebra, Number Theory, and Prealgebra.

### Extracting answers from MATH solutions

MATH stores answers inside `\boxed{...}` in the solution string. We need a helper to
extract them:

In [51]:
TOOL_SUBJECTS = [
    "Algebra", "Number Theory", "Prealgebra", "Intermediate Algebra",
]


def extract_boxed(solution):
    """Extract the content of the last \\boxed{...} in a MATH solution string."""
    idx = solution.rfind("\\boxed{")
    if idx == -1:
        return solution.strip()
    start = idx + len("\\boxed{")
    depth = 1
    i = start
    while i < len(solution) and depth > 0:
        if solution[i] == "{":
            depth += 1
        elif solution[i] == "}":
            depth -= 1
        i += 1
    return solution[start:i - 1].strip()


def record_to_sample(record):
    """Convert a MATH-500 record into an inspect_ai Sample."""
    target = record.get("answer") or extract_boxed(record["solution"])
    return Sample(
        input=record["problem"],
        target=target,
        metadata={
            "level": int(record["level"]),
            "subject": record["subject"],
        },
    )

In [52]:
full_dataset = hf_dataset(
    path="HuggingFaceH4/MATH-500",
    split="test",
    sample_fields=record_to_sample,
    cached=True,
)

print(f"Total MATH-500: {len(full_dataset)} samples")

subject_counts = defaultdict(int)
for s in full_dataset:
    subject_counts[s.metadata["subject"]] += 1

print(f"\n{'Subject':<30s} {'Count':>5s}")
print("-" * 37)
for subj in sorted(subject_counts):
    marker = " <-- tool-friendly" if subj in TOOL_SUBJECTS else ""
    print(f"{subj:<30s} {subject_counts[subj]:>5d}{marker}")

Loading dataset HuggingFaceH4/MATH-500 from Hugging Face...


README.md:   0%|          | 0.00/412 [00:00<?, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

Total MATH-500: 500 samples

Subject                        Count
-------------------------------------
Algebra                          124 <-- tool-friendly
Counting & Probability            38
Geometry                          41
Intermediate Algebra              97 <-- tool-friendly
Number Theory                     62 <-- tool-friendly
Prealgebra                        82 <-- tool-friendly
Precalculus                       56


## Dev/test split

A core principle in evaluation: **never tune on your test set.** Iterating on the same
data you use for final scoring inflates results and makes them meaningless. We split the
tool-friendly subset into a small **dev set** (10%) for prompt and scaffolding iteration,
and a larger **test set** (90%) reserved for final evaluation only.

This mirrors the elicitation workflow described in [METR's Guidelines for Capability Elicitation](https://evaluations.metr.org/elicitation-protocol/): iterate against the dev set until failures stabilize, then run the test set once.

> **Adjust to your hardware.** If even the dev set takes too long, reduce `split_point`
> or set a smaller `EVAL_LIMIT`. The point is rapid iteration — you can always increase
> the test set size later.

In [53]:
tool_dataset = [s for s in full_dataset if s.metadata["subject"] in TOOL_SUBJECTS]
print(f"Tool-friendly subset: {len(tool_dataset)} samples")

random.seed(RANDOM_SEED)
random.shuffle(tool_dataset)

split_point = int(len(tool_dataset) * 0.1)
DEV_SET = tool_dataset[:split_point]
TEST_SET = tool_dataset[split_point:]

print(f"DEV_SET:  {len(DEV_SET)} samples")
print(f"TEST_SET: {len(TEST_SET)} samples")

Tool-friendly subset: 365 samples
DEV_SET:  36 samples
TEST_SET: 329 samples


## 6. Scoring Mathematical Answers

For the toy problems we used `match(numeric=True)`, which works when answers are plain
numbers. But MATH answers can be fractions (`3/7`), expressions (`2\sqrt{5}`), or
formatted in different equivalent ways — a simple string match will miss many correct answers.

We covered `model_graded_qa()` in notebook 3. Here we apply it to math: the key challenge
is writing a grading prompt that handles equivalent notations (e.g. `1/2` vs `0.5` vs `\frac{1}{2}`).
Note that in notebook 3 we deliberately hid the reference answer from the grading model — here
there's no reason to do that, so you can pass the correct answer directly.

> **Trade-off:** Model-graded scoring is slower and noisier, but catches equivalences that
> string matching misses. For production evals you might want a more capable model for
> grading than the one being tested — think about what makes sense for your setup.

## Assignment 3: Define the math scorer

Define a `math_scorer` using `model_graded_qa()`. Write a grading prompt that instructs
the model to judge mathematical equivalence and respond with **C** (correct) or **I** (incorrect),
and choose which model should do the grading. Think about what edge cases matter: different
notations, equivalent fractions, simplified vs unsimplified forms.

In [ ]:
SCORER_MODEL = # YOUR CODE HERE 

GRADING_INSTRUCTIONS = # YOUR CODE HERE

MATH_SCORER = # YOUR CODE HERE

In [58]:
# Модель-судья — используем ту же, но можно поставить более сильную
# (например ollama/qwen2.5:7b), если она доступна
SCORER_MODEL = MODEL  # или "ollama/qwen2.5:7b" для более надёжной оценки


GRADING_INSTRUCTIONS = """
You are a math grader. Your task is to judge whether a student's answer
is mathematically equivalent to the correct answer.

Correct answer: {criterion}
Student's answer: {answer}

Grading rules:
- Mark C (correct) if the answers are mathematically equivalent, even if
  the format differs. Examples of equivalent pairs:
    5/2  =  2.5  =  2½  =  \\frac{{5}}{{2}}
    \\sqrt{{2}}  =  2**0.5  =  1.41421...  (when simplified form is expected)
    -3, 3  =  x=-3 and x=3  =  ±3
    0.333...  =  1/3  =  \\frac{{1}}{{3}}
- Mark C if the student gave an unsimplified but equivalent form (e.g. 4/8 for 1/2).
- Mark I (incorrect) if the numeric value differs, even slightly.
- Mark I if the student gave extra numbers not in the correct answer.
- Respond with ONLY the single letter C or I, nothing else.
""".strip()


MATH_SCORER = model_graded_qa(
    template=GRADING_INSTRUCTIONS,
    model=SCORER_MODEL,
    grade_pattern=r"(?i)^([CI])",  # ← скобки вокруг [CI]
    )


In [59]:
# Run the scorer on two toy samples where we know the answer.
_scorer_test_samples = [
    Sample(input="What is 1+1?", target="2"),
    Sample(input="What is 10/4?", target="5/2"),
]

_log_scorer_test = eval(
    Task(
        dataset=_scorer_test_samples,
        solver=[system_message("Answer the math question. Reply with just the answer."), generate()],
        scorer=MATH_SCORER,
    ),
    model=MODEL,
)[0]

print_results("Scorer sanity check", _log_scorer_test)
print("Check: does the scorer correctly mark equivalent answers as C?")

Output()

  Scorer sanity check   accuracy: 100%
  [C] # 1  msgs= 3  target=                   2  C
  [C] # 2  msgs= 3  target=                 5/2  C

Check: does the scorer correctly mark equivalent answers as C?


---
1. Did your scorer correctly handle the fraction equivalence (`10/4` vs `5/2`)?
2. What failure modes can you imagine for model-graded scoring?

**Your answer:**

- справился хорошо 

- хорший вопрос, ПОДУМАТЬ НАД ВТОРЫМ ВОПРОСОМ 

## 7. Dev-Set Iteration — Building and Improving Your Agent

This is the core of agent evaluation: run on the dev set, see where the agent struggles,
and systematically improve it. The iteration loop:

1. Run the current agent on the dev set
2. Inspect failures — *why* did the agent get these wrong?
3. Hypothesize an improvement (better prompt? more tools? output formatting?) — and check
   whether the answer was actually correct but the scorer marked it wrong
4. Implement the change and re-evaluate on the dev set
5. Repeat

## Assignment 4: Build and improve a ReAct agent

Your goal is to build a ReAct agent and iterate on it using the dev set.
Does adding tools and scaffolding help on this task, and by how much?
Start by running a basic ReAct agent on the dev set, then look at the failures in the logs and iterate.

**Step 1 — Run a baseline.** Write a system prompt and run `react()` with `ALL_TOOLS` on the dev set.

**Step 2 — Inspect failures.** Open the logs and look at what went wrong. Common things to consider:
- Is the model ignoring tools and is failing?
- Is the answer mathematically correct but formatted wrong (e.g. `0.5` instead of `1/2`)?
- Is the model getting lost in multi-step problems?

**Step 3 — Iterate.** Based on what you see, try to improve. Some directions:
- Make the system prompt more explicit about strategy or answer format
- Add tools that cover operations the model struggles with (e.g. a single `calculator` tool, `gcd`, `factorial` or something very different)
- Add a format-extraction step after the `react()` loop if formatting is the main issue
- Reconsider the scorer if correct answers are being marked wrong

For each configuration you try, store the result as a `(description, log)` tuple in `DEV_RUNS` at the bottom — this will let you compare all your attempts in one table. Try at least two configurations.

In [60]:
MY_REACT_PROMPT = """
You are a precise math solver competing in a math olympiad.
You have access to tools — use them for ALL arithmetic and algebra.

Tools available:
- calculator(expression)      — evaluate any Python math expression
- modular_arithmetic(a, b)    — compute a mod b
- sympy_solve(equation)       — solve equations for x symbolically
- quadratic_solver(a, b, c)   — solve a*x^2 + b*x + c = 0
- prime_check(n)              — check if n is prime
- list_factors(n)             — list all factors of n

Strategy:
1. Read the problem and identify what type of computation is needed.
2. Call the appropriate tool for EACH arithmetic step — never compute mentally.
3. Once you have the final result, call submit() with ONLY the answer.

Answer format rules:
- Submit bare numbers or expressions: 42, 3/4, sqrt(2), -7
- Do NOT include: "x =", units, LaTeX (\\boxed etc.), explanation
- For fractions, use the simplified form: 1/2 not 2/4
""".strip()


log_attempt_1 = eval(
    Task(
        dataset=DEV_SET,
        solver=react(
            prompt=MY_REACT_PROMPT,
            tools=ALL_TOOLS,
            attempts=1,
        ),
        scorer=MATH_SCORER,
        message_limit=MAX_MESSAGES,
    ),
    model=MODEL,
    limit=EVAL_LIMIT,
)[0]


print_results("Attempt 1 (baseline)", log_attempt_1)
DEV_RUNS = [("Attempt 1 (baseline)", log_attempt_1)]

Output()

  Attempt 1 (baseline)   accuracy: 70%
  [C] # 1  msgs= 5  target=                  10  C
  [C] # 2  msgs=20  target=                  47  C
  [C] # 3  msgs= 7  target=                2k+2  C
  [C] # 4  msgs=20  target=                   2  C
  [I] # 5  msgs=20  target=                   5  I
  [C] # 6  msgs= 7  target=          (a+5)(b+2)  C
  [C] # 7  msgs= 8  target=                 550  C
  [C] # 8  msgs= 3  target=                   2  C
  [C] # 9  msgs= 5  target=                   0  C
  [C] #10  msgs= 4  target=                   1  C
  [I] #11  msgs= 9  target=                   3  I
  [C] #12  msgs= 3  target=         \frac{3}{2}  C
  [I] #13  msgs= 7  target=                   1  I
  [I] #14  msgs=20  target=                   4  I
  [C] #15  msgs= 7  target=                  14  C
  [C] #16  msgs= 9  target=               26000  C
  [C] #17  msgs=20  target=          \$32,\!348  C
  [C] #18  msgs= 8  target=                  13  C
  [C] #19  msgs=10  target=                

In [61]:
# v2

@tool
def multiplicative_order():
    async def execute(a: int, n: int) -> str:
        """Find the multiplicative order of a modulo n.

        Returns the smallest positive integer k such that a^k ≡ 1 (mod n).
        Useful for repeating decimal periods, cyclic group problems, and
        number theory tasks involving powers and modular arithmetic.

        Examples:
            multiplicative_order(10, 7)     → "6"
            multiplicative_order(10, 11111) → "5"

        Args:
            a: The base integer.
            n: The modulus (must be positive and coprime with a).

        Returns:
            The multiplicative order as a string, or an error message.
        """
        import math
        if n <= 1:
            return "Error: n must be greater than 1."
        if math.gcd(a, n) != 1:
            return f"Error: gcd({a}, {n}) != 1 — multiplicative order does not exist."
        k = 1
        current = a % n
        while current != 1:
            current = (current * a) % n
            k += 1
            if k > n:
                return "Error: order not found within n steps."
        return str(k)
    return execute

In [63]:
# ALL_TOOLS_V2 = [
#     calculator(),
#     modular_arithmetic(),
#     sympy_solve(),
#     quadratic_solver(),
#     prime_check(),
#     list_factors(),
#     multiplicative_order(),   # новый
# ]

MY_REACT_PROMPT_V2 = """
You are a precise math solver competing in a math olympiad.
You have access to tools — use them for ALL arithmetic and algebra.

Tools available:
- calculator(expression)          — evaluate any Python math expression
- modular_arithmetic(a, b)        — compute a mod b
- multiplicative_order(a, n)      — smallest k such that a^k ≡ 1 (mod n)
                                    use this for repeating decimal period problems
- sympy_solve(equation)           — solve equations for x symbolically
- quadratic_solver(a, b, c)       — solve a*x^2 + b*x + c = 0
- prime_check(n)                  — check if n is prime
- list_factors(n)                 — list all factors of n

Strategy:
1. Identify the computation type and pick the right tool.
2. Call tools for EACH arithmetic step — never compute mentally.
3. When you have the final answer, call the submit() tool.

CRITICAL — submit rules:
- You MUST call submit() using the tool-call interface, NOT as plain text.
  WRONG: writing "submit(42)" in your message
  RIGHT: using the structured tool call button for submit
- Submit ONLY the bare answer: 42, 3/4, 13/4, sqrt(2), -7
- No LaTeX (\\boxed, \\frac), no "x =", no units, no explanation.
""".strip()


log_attempt_2 = eval(
    Task(
        dataset=DEV_SET,
        solver=react(
            prompt=MY_REACT_PROMPT_V2,
            # tools=ALL_TOOLS_V2,
            attempts=1,
        ),
        scorer=MATH_SCORER,
        message_limit=MAX_MESSAGES,
    ),
    model=MODEL,
    limit=EVAL_LIMIT,
)[0]


print_results("Attempt 2 (+ multiplicative_order + submit fix)", log_attempt_2)
DEV_RUNS.append(("Attempt 2 (+ multiplicative_order + submit fix)", log_attempt_2))

Output()

  Attempt 2 (+ multiplicative_order + submit fix)   accuracy: 67%
  [C] # 1  msgs=20  target=                  10  C
  [C] # 2  msgs=20  target=                  47  C
  [C] # 3  msgs= 5  target=                2k+2  C
  [I] # 4  msgs= 3  target=                   2  I
  [I] # 5  msgs=20  target=                   5  I
  [C] # 6  msgs= 3  target=          (a+5)(b+2)  C
  [I] # 7  msgs=17  target=                 550  I
  [I] # 8  msgs=18  target=                   2  I
  [C] # 9  msgs= 3  target=                   0  C
  [C] #10  msgs= 7  target=                   1  C
  [I] #11  msgs= 4  target=                   3  I
  [C] #12  msgs= 5  target=         \frac{3}{2}  C
  [I] #13  msgs= 5  target=                   1  I
  [C] #14  msgs=20  target=                   4  C
  [C] #15  msgs= 9  target=                  14  C
  [C] #16  msgs=13  target=               26000  C
  [C] #17  msgs=20  target=          \$32,\!348  C
  [C] #18  msgs= 3  target=                  13  C
  [C] #19  msgs=

@tool
def add():
    async def execute(a: float, b: float) -> str:
        """Add two numbers.

        Args:
            a: First number.
            b: Second number.
        """
        try:
            return str(float(a) + float(b))
        except Exception as e:
            return f"Error: {e}"
    return execute


@tool
def subtract():
    async def execute(a: float, b: float) -> str:
        """Subtract b from a.

        Args:
            a: Number to subtract from.
            b: Number to subtract.
        """
        try:
            return str(float(a) - float(b))
        except Exception as e:
            return f"Error: {e}"
    return execute


@tool
def multiply():
    async def execute(a: float, b: float) -> str:
        """Multiply two numbers.

        Args:
            a: First number.
            b: Second number.
        """
        try:
            return str(float(a) * float(b))
        except Exception as e:
            return f"Error: {e}"
    return execute


@tool
def divide():
    async def execute(a: float, b: float) -> str:
        """Divide a by b.

        Args:
            a: Dividend.
            b: Divisor (must not be zero).
        """
        try:
            b_val = float(b)
            if b_val == 0:
                return "Error: division by zero."
            return str(float(a) / b_val)
        except Exception as e:
            return f"Error: {e}"
    return execute

In [64]:
ARITH_TOOLS = [
    add(),
    subtract(),
    multiply(),
    divide(),
    modular_arithmetic(),
    multiplicative_order(),
]

MY_REACT_PROMPT_V3 = """
You are a precise math solver. Use tools for ALL computation — never calculate mentally.

TOOLS:
- add(a, b)                    — a + b
- subtract(a, b)               — a - b
- multiply(a, b)               — a * b
- divide(a, b)                 — a / b
- modular_arithmetic(a, b)     — remainder of a divided by b (a mod b)
- multiplicative_order(a, n)   — smallest k such that a^k ≡ 1 (mod n)
                                 use this for repeating decimal period problems

TOOL SELECTION GUIDE:
- Arithmetic (+, -, *, /)      → add / subtract / multiply / divide
- Remainder / clock problems   → modular_arithmetic
- Repeating decimal period     → multiplicative_order(10, denominator)
- Multi-step problems          → chain tool calls step by step

STRATEGY:
1. Break the problem into single-operation steps.
2. Call one tool per step, observe the result, then proceed.
3. For equations (e.g. "find x where 3x + 7 = 22"):
   - Rearrange manually: x = (22 - 7) / 3
   - Then call subtract(22, 7) → divide(result, 3)

SUBMIT — CRITICAL:
You MUST use submit() as a TOOL CALL, not as plain text.
  ✗ WRONG: writing "submit(42)" anywhere in your message text
  ✓ RIGHT: using the structured tool call interface for submit()
Submit ONLY the bare answer: 42, 13/4, 3/2, -7
No LaTeX (\\frac, \\boxed), no "x =", no units.
""".strip()

MY_ON_CONTINUE_V3 = (
    "Please continue working on the problem. "
    "If you have the final answer, call submit() using the TOOL CALL INTERFACE — "
    "writing submit() as plain text in your message does nothing."
)

log_attempt_3 = eval(
    Task(
        dataset=DEV_SET,
        solver=react(
            prompt=MY_REACT_PROMPT_V3,
            tools=ARITH_TOOLS,
            on_continue=MY_ON_CONTINUE_V3,
            attempts=1,
        ),
        scorer=MATH_SCORER,
        message_limit=MAX_MESSAGES,
    ),
    model=MODEL,
    limit=EVAL_LIMIT,
)[0]

print_results("Attempt 3 (correct tools + on_continue + submit fix)", log_attempt_3)
DEV_RUNS.append(("Attempt 3 (correct tools + on_continue)", log_attempt_3))

Output()

  Attempt 3 (correct tools + on_continue + submit fix)   accuracy: 63%
  [I] # 1  msgs= 3  target=                  10  I
  [C] # 2  msgs= 3  target=                  47  C
  [C] # 3  msgs= 5  target=                2k+2  C
  [I] # 4  msgs= 5  target=                   2  I
  [I] # 5  msgs= 5  target=                   5  I
  [C] # 6  msgs= 3  target=          (a+5)(b+2)  C
  [C] # 7  msgs= 7  target=                 550  C
  [C] # 8  msgs= 3  target=                   2  C
  [C] # 9  msgs= 3  target=                   0  C
  [C] #10  msgs= 5  target=                   1  C
  [C] #11  msgs= 5  target=                   3  C
  [C] #12  msgs=20  target=         \frac{3}{2}  C
  [I] #13  msgs= 3  target=                   1  I
  [C] #14  msgs= 3  target=                   4  C
  [C] #15  msgs= 3  target=                  14  C
  [C] #16  msgs=10  target=               26000  C
  [I] #17  msgs=20  target=          \$32,\!348  I
  [C] #18  msgs= 3  target=                  13  C
  [I] #19  

In [65]:
MY_REACT_PROMPT_V4 = """
You are a precise math solver.

Use tools ONLY when you need to compute with specific numbers.
Do NOT use tools for symbolic algebra, factoring, or geometry — solve those directly.

TOOLS (for numeric computation only):
- add(a, b)                  — a + b
- subtract(a, b)             — a - b  
- multiply(a, b)             — a * b
- divide(a, b)               — a / b
- modular_arithmetic(a, b)   — a mod b
- multiplicative_order(a, n) — smallest k: a^k ≡ 1 (mod n)
                               use for repeating decimal period problems

WHEN TO USE TOOLS:
  ✓ Large number arithmetic: multiply(4738, 2659)
  ✓ Remainders: modular_arithmetic(1000000, 397)
  ✓ Decimal periods: multiplicative_order(10, 11111)
  ✗ Factoring expressions: solve directly without tools
  ✗ Geometry / angles: reason directly
  ✗ Simple equations with small numbers: reason directly

SUBMIT:
Call submit() using the TOOL CALL INTERFACE with the bare answer.
Examples: 42  |  3/2  |  (a+5)(b+2)  |  36  |  13/4
No LaTeX, no "x =", no units.
""".strip()

log_attempt_4 = eval(
    Task(
        dataset=DEV_SET,
        solver=react(
            prompt=MY_REACT_PROMPT_V4,
            tools=ARITH_TOOLS,
            on_continue=MY_ON_CONTINUE_V3,
            attempts=1,
        ),
        scorer=MATH_SCORER,
        message_limit=MAX_MESSAGES,
    ),
    model=MODEL,
    limit=EVAL_LIMIT,
)[0]

print_results("Attempt 4 (selective tool use)", log_attempt_4)
DEV_RUNS.append(("Attempt 4 (selective tool use)", log_attempt_4))

Output()

  Attempt 4 (selective tool use)   accuracy: 77%
  [C] # 1  msgs=20  target=                  10  C
  [I] # 2  msgs= 5  target=                  47  I
  [C] # 3  msgs= 6  target=                2k+2  C
  [I] # 4  msgs=11  target=                   2  I
  [C] # 5  msgs= 5  target=                   5  C
  [C] # 6  msgs= 3  target=          (a+5)(b+2)  C
  [C] # 7  msgs= 8  target=                 550  C
  [C] # 8  msgs= 3  target=                   2  C
  [C] # 9  msgs= 3  target=                   0  C
  [C] #10  msgs= 3  target=                   1  C
  [C] #11  msgs= 3  target=                   3  C
  [C] #12  msgs= 3  target=         \frac{3}{2}  C
  [I] #13  msgs= 3  target=                   1  I
  [C] #14  msgs= 5  target=                   4  C
  [C] #15  msgs= 3  target=                  14  C
  [C] #16  msgs= 8  target=               26000  C
  [I] #17  msgs= 6  target=          \$32,\!348  I
  [C] #18  msgs= 4  target=                  13  C
  [C] #19  msgs= 3  target=      

In [ ]:
# Look at the failures in the logs and decide what to change:
# prompt, tools, scorer, add a format-extraction step after react() or something else.
# Duplicate this cell for each new configuration. Give each log a new name.

# YOUR CODE HERE: define your updated prompt, tools, scorer, or solver

log_attempt_2 = eval(
    Task(
        dataset=DEV_SET,
        solver=# YOUR CODE HERE,
        scorer=# YOUR CODE HERE,
        message_limit=MAX_MESSAGES,
    ),
    model=MODEL,
    limit=EVAL_LIMIT,
)[0]

description_2 = # YOUR CODE HERE
print_results(description_2, log_attempt_2)
DEV_RUNS.append((description_2, log_attempt_2))

In [67]:
# --- 1. Улучшенный скорер ---
GRADING_INSTRUCTIONS_V2 = """
You are a math grader. Judge if the student's answer is mathematically equivalent to the correct answer.

Correct answer: {criterion}
Student's answer: {answer}

Equivalence rules — mark C if ANY of these hold:
- Numeric equality: 3.25 = 13/4 = \\frac{{13}}{{4}}
- Dollar amounts: 32348 = $32,348 = 32,348 = \\$32,\\!348
- Fractions: 17/21 = \\frac{{17}}{{21}}, 3/2 = 1.5 = \\frac{{3}}{{2}}
- Expressions: (a+5)(b+2) = (b+2)(a+5)
- Angles: 36 = 36^\\circ (when context is degrees)
- Simplified vs unsimplified: 4/8 = 1/2

Mark I only if the numeric value genuinely differs.
Respond with ONLY the single letter C or I.
""".strip()

MATH_SCORER_V2 = model_graded_qa(
    template=GRADING_INSTRUCTIONS_V2,
    model=SCORER_MODEL,
    grade_pattern=r"(?i)^([CI])",
)

# --- 2. Format-extraction step после react() ---
FORMAT_PROMPT = (
    "The answer to the math problem has been found above. "
    "Extract ONLY the final answer — a bare number or expression. "
    "No LaTeX, no explanation, no units. Examples: 42, 13/4, 3/2, (a+5)(b+2)"
)

log_attempt_5 = eval(
    Task(
        dataset=DEV_SET,
        solver=[
            react(
                prompt=MY_REACT_PROMPT_V4,
                tools=ARITH_TOOLS,
                on_continue=MY_ON_CONTINUE_V3,
                attempts=1,
            ),
            system_message(FORMAT_PROMPT),
            generate(),
        ],
        scorer=MATH_SCORER_V2,
        message_limit=MAX_MESSAGES,
    ),
    model=MODEL,
    limit=EVAL_LIMIT,
)[0]

print_results("Attempt 5 (scorer v2 + format extraction)", log_attempt_5)
DEV_RUNS.append(("Attempt 5 (scorer v2 + format extraction)", log_attempt_5))

Output()

  Attempt 5 (scorer v2 + format extraction)   accuracy: 53%
  [C] # 1  msgs=20  target=                  10  C
  [I] # 2  msgs=14  target=                  47  I
  [C] # 3  msgs= 5  target=                2k+2  C
  [I] # 4  msgs=15  target=                   2  I
  [C] # 5  msgs=20  target=                   5  C
  [C] # 6  msgs= 5  target=          (a+5)(b+2)  C
  [I] # 7  msgs=11  target=                 550  I
  [I] # 8  msgs= 5  target=                   2  I
  [I] # 9  msgs= 5  target=                   0  I
  [C] #10  msgs=20  target=                   1  C
  [I] #11  msgs= 5  target=                   3  I
  [C] #12  msgs=20  target=         \frac{3}{2}  C
  [I] #13  msgs= 7  target=                   1  I
  [I] #14  msgs=14  target=                   4  I
  [I] #15  msgs= 5  target=                  14  I
  [I] #16  msgs=10  target=               26000  I
  [C] #17  msgs=11  target=          \$32,\!348  C
  [C] #18  msgs=20  target=                  13  C
  [C] #19  msgs=11  ta

In [68]:
print(f"\n{'Attempt':<45s}  {'Acc':>5s}  {'Avg msgs':>8s}")
print("-" * 62)
for label, log in DEV_RUNS:
    acc = get_acc(log)
    avg_m = sum(len(s.messages) for s in log.samples) / len(log.samples)
    print(f"{label:<45s}  {acc:>5.0%}  {avg_m:>8.1f}")


Attempt                                          Acc  Avg msgs
--------------------------------------------------------------
Attempt 1 (baseline)                             70%      11.6
Attempt 2 (+ multiplicative_order + submit fix)    67%       9.7
Attempt 3 (correct tools + on_continue)          63%       6.1
Attempt 4 (selective tool use)                   77%       7.4
Attempt 5 (scorer v2 + format extraction)        53%      11.5


In [ ]:
print(f"{'Configuration':<40s} {'Dev Accuracy':>12s}")
print("=" * 54)
for description, log in DEV_RUNS:
    acc = get_acc(log)
    print(f"{description:<40s} {acc:>12.0%}")

---
1. What modifications did you try? Which had the biggest impact?
2. What was the best dev-set accuracy you achieved? What configuration produced it?
3. Did any change that you expected to help actually hurt? Why might that be?
4. Look at the individual failures. What are the most common error types?

**Your answer:**



**Your answer:**

**1. Какие модификации пробовали? Что дало наибольший эффект?**

Протестировано 4 конфигурации на dev set. Наибольший эффект дала **Attempt 4: избирательное использование инструментов** — изменение промпта с `"используй инструменты для ВСЕХ вычислений"` на `"используй инструменты ТОЛЬКО для работы с конкретными числами"`. Это одно изменение подняло точность с 70% до 77%. Добавление `multiplicative_order` закрыло задачи на теорию чисел (период повторяющейся дроби), которые ни один из существующих инструментов решить не мог. Улучшенный скорер v2 исправил ошибки эквивалентности для ответов в LaTeX-формате (`\$32,\!348` и `\frac{17}{21}`).

**2. Лучшая точность и конфигурация?**

Лучший результат: **77% — Attempt 4** (`MY_REACT_PROMPT_V4` с избирательным использованием инструментов + `ARITH_TOOLS` включая `multiplicative_order` + `on_continue` + `MATH_SCORER`).

**3. Было ли что-то что должно было помочь, но навредило?**

Да — **Attempt 5 (format-extraction шаг) упал до 53%**, несмотря на улучшенный скорер. Добавление `system_message() + generate()` после `react()` создало второй проход генерации поверх уже завершённого трейса. Модель теряла контекст и начинала заново решать задачи которые уже решила правильно, внося новые ошибки. Вывод: добавление второго `generate()` после `react()` — это не чистая постобработка формата, а повторный запуск рассуждений. Attempt 3 также навредил (63% vs 70%) потому что `tools=ALL_TOOLS` был случайно закомментирован и модель работала вообще без инструментов.

**4. Основные типы ошибок в оставшихся неверных примерах?**

Три паттерна:

- **Submit loop** (`msgs=20`): модель пишет `submit(3)` как plain text снова и снова вместо structured tool call. Это фундаментальное ограничение `qwen2.5:latest` — модель не умеет надёжно отличать написание текста от вызова инструмента. Сэмплы #12, #21, #22 столкнулись с этой проблемой.

- **Неверный ответ на символьных/рассуждательных задачах** (`msgs=3–5`): сэмплы #2, #4, #13 — модель отвечает сразу без инструментов и ошибается в логике. Это задачи где никакой арифметический инструмент не поможет; узкое место — чистое математическое рассуждение которого у модели такого размера просто нет.

- **Несоответствие формата для дробей** (`#23`, `#28`): модель возвращает `3.25` когда ожидается `13/4`. Скорер v2 частично исправляет это, но только когда используется без сломанного format-extraction шага.

## 8. Test-Set Evaluation

You've iterated on the dev set and found your best configuration. Now it's time for the
moment of truth: evaluating on the held-out test set.

> **Run this section only once**, with your best configuration. Re-running and picking
> the best result would be "test-set contamination" — the same as peeking at a test set
> in ML.

## Assignment 5: Run and analyze your best configuration on the test set

Run your best agent configuration on the held-out test set, report accuracy with a 95%
confidence interval, and break down performance by subject and difficulty level. Note
anything that stands out.

In [69]:
def wilson_ci(n_correct, n_total, confidence_level=0.95):
    """Wilson score interval for a binomial proportion."""
    z = norm.ppf(0.5 + confidence_level / 2)
    p_hat = n_correct / n_total
    denom = 1 + z ** 2 / n_total
    centre = (p_hat + z ** 2 / (2 * n_total)) / denom
    margin = z * math.sqrt(
        (p_hat * (1 - p_hat) + z ** 2 / (4 * n_total)) / n_total
    ) / denom
    return max(0, centre - margin), min(1, centre + margin)

## Assignment 5.1: Run on the test set

Use your best configuration and run it to evaluate the test set

In [ ]:
log_test = eval(
    Task(
        dataset=TEST_SET,
        # YOUR CODE HERE
    ),
    model=MODEL,
    limit=len(TEST_SET),
)[0]

n_test = # YOUR CODE HERE
n_correct_test = # YOUR CODE HERE

lo, hi = wilson_ci(n_correct_test, n_test)
print(f"Test accuracy : {n_correct_test / n_test:.1%}")
print(f"95% Wilson CI : [{lo:.1%}, {hi:.1%}]")
print(f"n = {n_test}")

```
# Запускаем лучшую конфигурацию (Attempt 4) на TEST_SET — один раз!
log_test = eval(
    Task(
        dataset=TEST_SET,
        solver=react(
            prompt=MY_REACT_PROMPT_V4,
            tools=ARITH_TOOLS,
            on_continue=MY_ON_CONTINUE_V3,
            attempts=1,
        ),
        scorer=MATH_SCORER,
        message_limit=MAX_MESSAGES,
    ),
    model=MODEL,
    limit=len(TEST_SET),
)[0]

# Считаем метрики
n_test = len(log_test.samples)
n_correct_test = sum(1 for s in log_test.samples if _first_score(s).value == 1)

lo, hi = wilson_ci(n_correct_test, n_test)
print(f"Test accuracy : {n_correct_test / n_test:.1%}")
print(f"95% Wilson CI : [{lo:.1%}, {hi:.1%}]")
print(f"n = {n_test}")
```

In [70]:
log_test = eval(
    Task(
        dataset=TEST_SET,
        solver=react(
            prompt=MY_REACT_PROMPT_V4,
            tools=ARITH_TOOLS,
            on_continue=MY_ON_CONTINUE_V3,
            attempts=1,
        ),
        scorer=MATH_SCORER,
        message_limit=MAX_MESSAGES,
    ),
    model=MODEL,
    limit=len(TEST_SET),
)[0]

# Считаем метрики
n_test = len(log_test.samples)
n_correct_test = sum(1 for s in log_test.samples if _first_score(s).value == 1)

lo, hi = wilson_ci(n_correct_test, n_test)
print(f"Test accuracy : {n_correct_test / n_test:.1%}")
print(f"95% Wilson CI : [{lo:.1%}, {hi:.1%}]")
print(f"n = {n_test}")




Output()

Test accuracy : 0.0%
95% Wilson CI : [0.0%, 1.2%]
n = 329


## Assignment 5.2: Breakdown by subject and difficulty level

There aren't enough samples per category to draw firm conclusions — that's fine.
Just see if anything stands out.

Iterate over logs and produce **two separate tables**: one grouped
by subject, one by difficulty level. 

Your output should look like this:

**By subject:**

| Subject | Correct | Total | Acc |
|---|---:|---:|---:|
| Algebra | 1 | 10 | 10% |
| Precalculus | 3 | 6 | 50% |
| ... | | | |

**By level:**

| Level | Correct | Total | Acc |
|---|---:|---:|---:|
| 1 | 4 | 5 | 80% |
| 2 | 3 | 6 | 50% |
| ... | | | |

The actual numbers will appear in the cell below.

In [73]:
subject_stats = defaultdict(lambda: [0, 0])
level_stats = defaultdict(lambda: [0, 0])

for sample in log_test.samples:
    sc = _first_score(sample)
    correct = 1 if sc.value == "C" else 0
    subj = sample.metadata.get("subject", "unknown")
    lvl = sample.metadata.get("level", 0)
    subject_stats[subj][0] += correct
    subject_stats[subj][1] += 1
    level_stats[lvl][0] += correct
    level_stats[lvl][1] += 1

print(f"{'Subject':<30s} {'Correct':>7s} {'Total':>5s} {'Acc':>6s}")
print("-" * 50)
for subj in sorted(subject_stats):
    c, t = subject_stats[subj]
    print(f"{subj:<30s} {c:>7d} {t:>5d} {c/t:>6.0%}")

print()
print(f"{'Level':<30s} {'Correct':>7s} {'Total':>5s} {'Acc':>6s}")
print("-" * 50)
for lvl in sorted(level_stats):
    c, t = level_stats[lvl]
    print(f"Level {lvl:<24d} {c:>7d} {t:>5d} {c/t:>6.0%}")

Subject                        Correct Total    Acc
--------------------------------------------------
Algebra                             81   113    72%
Intermediate Algebra                28    85    33%
Number Theory                       30    58    52%
Prealgebra                          43    73    59%

Level                          Correct Total    Acc
--------------------------------------------------
Level 1                             25    32    78%
Level 2                             46    59    78%
Level 3                             42    63    67%
Level 4                             40    85    47%
Level 5                             29    90    32%


**Your results:**

Fill in the tables once you've run the cell above.

**By subject:**

| Subject | Correct | Total | Acc |
|---|---:|---:|---:|
| Algebra | | | |
| Precalculus | | | |
| ... | | | |

**By difficulty level:**

| Level | Correct | Total | Acc |
|---|---:|---:|---:|
| 1 | | | |
| 2 | | | |
| ... | | | |

## Final comparison: dev vs test

In [79]:
# print(f"{'Configuration':<40s} {'Dev Acc':>8s}  {'Test Acc':>8s}")
# print("=" * 50)
# for name, log in DEV_RUNS:
#     acc = get_acc(log)
#     print(f"{name:<40s} {acc:>8.0%}  {'--':>8s}")
# print(f"{'best agent (TEST)':<40s} {'--':>8s}  {n_correct_test / n_test:>8.1%}")
# print(f"\n95% CI on test: [{lo:.1%}, {hi:.1%}]")

# Пересчитываем из subject_stats чтобы гарантированно взять правильные значения
n_correct_test = sum(v[0] for v in subject_stats.values())
n_test         = sum(v[1] for v in subject_stats.values())

lo, hi = wilson_ci(n_correct_test, n_test)

print(f"{'Configuration':<45s} {'Dev Acc':>8s}  {'Test Acc':>9s}")
print("=" * 65)
for name, log in DEV_RUNS:
    acc = get_acc(log)
    print(f"{name:<45s} {acc:>8.0%}  {'--':>9s}")

test_acc = n_correct_test / n_test
print(f"{'Attempt 4 — best agent (TEST)':<45s} {'77%':>8s}  {test_acc:>9.1%}")
print(f"\n95% Wilson CI on test: [{lo:.1%}, {hi:.1%}]  (n={n_test})")

Configuration                                  Dev Acc   Test Acc
Attempt 1 (baseline)                               70%         --
Attempt 2 (+ multiplicative_order + submit fix)      67%         --
Attempt 3 (correct tools + on_continue)            63%         --
Attempt 4 (selective tool use)                     77%         --
Attempt 5 (scorer v2 + format extraction)          53%         --
Attempt 4 — best agent (TEST)                      77%      55.3%

95% Wilson CI on test: [49.9%, 60.6%]  (n=329)


---
1. How does the test accuracy compare to the dev accuracy? If there's a gap,
   what might explain it?
2. Which subjects and difficulty levels does the agent handle best? Worst?
3. Look at the confidence interval. Is it narrow enough to be useful, or would
   you want more test samples?
4. If you were to improve this agent further, where would you focus?

**Your answer:**

- точность на dev and test есть разница, из-за того что мы в основном отрабатывали для dev 
- что хорошо и что плохо 

лучше всего себя показал 3 вариант (максчимальная проработка , на фоне одного прохода)

в то же время att 5 показал уже себя плохо, возможно что второе размышление после первого нужно было применить другой промпт (ВТОРОЙ ПРОМПТ ДЛЯ ТОГО ЧТО БЫ МОДЕЛЬ РЕШАЛА НЕ ЗАНОВОГО , А ПРОЕРЯЛА ОТВЕТ ПЕРВОГО ПРОХОДА )

- 


- как улучшить ?

как мне кажеться что тут нужно будет изучить сам бенч чтобы сделать инструменты более подходящие для решения мат задач ( ну или уже найти готовые)

решение с двумя react звучит хорошо, нужно будет позаниматься промтингом 

**Your answer:**

**1. How does test accuracy compare to dev accuracy? If there's a gap, what might explain it?**

Dev accuracy составила 77% (Attempt 4), тест — 55.3%. Разрыв в ~22pp объясняется **prompt overfitting**: мы итерировали промпт 5 раз на одних и тех же 30 сэмплах dev-set, постепенно подстраивая его под конкретные паттерны ошибок этой выборки. На новых задачах из test-set те же промпт-решения уже не работают так хорошо. Это классическая проблема малого dev-set — 30 примеров недостаточно чтобы изменения в промпте были статистически обоснованы, а не случайны.

**2. Лучшие и худшие subjects и levels?**

Лучше всего: **Algebra (72%)** и **Level 1–2 (78%)** — задачи решаются прямым символьным рассуждением или простой арифметикой, инструменты дают ощутимый буст. Хуже всего: **Intermediate Algebra (33%)** и **Level 5 (32%)** — здесь требуется работа с многочленами, комплексными числами, системами уравнений. Наши инструменты (`add/subtract/multiply/divide`) для таких задач почти бесполезны. Level 5 вплотную приближается к случайному угадыванию, что говорит о **hard capability ceiling** модели `qwen2.5:3b` на олимпийском уровне сложности.

**3. Достаточно ли узок доверительный интервал?**

CI: [49.9%, 60.6%] при n=329 — ширина интервала **~11pp**. Для грубых выводов (модель работает / не работает) этого достаточно. Но для тонких сравнений — например, "Attempt 4 лучше Attempt 3 на 14pp на dev-set" — такой интервал слишком широк чтобы утверждать что разница значима. Для надёжных выводов на уровне отдельных subjects нужно минимум 100–200 примеров на каждый subject (сейчас у Intermediate Algebra n=85, у Number Theory n=58 — маловато). Если бы это был реальный elicitation pipeline, стоило бы увеличить test-set до 500–1000 примеров.

**4. Как улучшить агента дальше?**

Три направления по приоритету:

- **Лучшие инструменты под бенч.** Нужно изучить какие типы задач встречаются в MATH-500 и добавить `sympy`-based инструменты: символьное упрощение, работа с многочленами, тригонометрия. `add/subtract/multiply` закрывают только простую арифметику.

- **Двухпроходная архитектура с правильным промптом.** Attempt 5 провалился потому что второй `generate()` перезапускал рассуждение с нуля. Правильная идея: первый `react()` решает задачу, второй проход получает промпт **"verify: is this answer correct? Check the calculation without re-solving"** — то есть верификация, а не повторное решение.

- **Более сильная модель для сложных задач.** Level 5 и Intermediate Algebra — это не проблема scaffolding, это проблема capability. `qwen2.5:3b` имеет hard ceiling который промпт-инжинирингом не обойти. `qwen2.5:7b` или `14b` дали бы значительный прирост именно на Level 4–5.

In [80]:
log_test

{
  "version": 2,
  "status": "success",
  "eval": {
    "eval_id": "Fq9eMwXRhUepQMnkG7EQK2",
    "run_id": "iuMKkeWsxvhPRnJZ2YALSz",
    "created": "2026-04-14T19:01:37+00:00",
    "task": "task",
    "task_id": "6gCpWFDqCvTLV7PWP6Yphi",
    "task_version": 0,
    "task_display_name": "task",
    "task_attribs": {},
    "task_args": {},
    "task_args_passed": {},
    "solver_args_passed": {},
    "dataset": {
      "samples": 329,
      "sample_ids": [
        1,
        2,
        3,
        4,
        5,
        6,
        7,
        8,
        9,
        10,
        11,
        12,
        13,
        14,
        15,
        16,
        17,
        18,
        19,
        20,
        21,
        22,
        23,
        24,
        25,
        26,
        27,
        28,
        29,
        30,
        31,
        32,
        33,
        34,
        35,
        36,
        37,
        38,
        39,
        40,
        41,
        42,
        43,
        44,
        45,
        46

In [78]:
n_test = len(log_test.samples)
n_correct_test = sum(1 for s in log_test.samples if _first_score(s).value == "C")

lo, hi = wilson_ci(n_correct_test, n_test)
print(f"Test accuracy : {n_correct_test / n_test:.1%}")
print(f"95% Wilson CI : [{lo:.1%}, {hi:.1%}]")
print(f"n = {n_test}")

# Breakdown по subject и level
subj_stats = defaultdict(lambda: {"correct": 0, "total": 0})
lvl_stats  = defaultdict(lambda: {"correct": 0, "total": 0})

for sample in log_test.samples:
    sc   = _first_score(sample)
    subj = sample.metadata.get("subject", "Unknown")
    lvl  = sample.metadata.get("level", "?")
    subj_stats[subj]["total"]   += 1
    subj_stats[subj]["correct"] += int(sc.value == "C")  # <-- исправлено
    lvl_stats[lvl]["total"]     += 1
    lvl_stats[lvl]["correct"]   += int(sc.value == "C")  # <-- исправлено

print("\nBy subject:")
print(f"  {'Subject':<30s}  {'Acc':>5s}  {'k/n':>7s}")
print("  " + "-" * 46)
for subj, d in sorted(subj_stats.items()):
    acc = d["correct"] / d["total"] if d["total"] else 0
    print(f"  {subj:<30s}  {acc:>5.0%}  {d['correct']:>3d}/{d['total']}")

print("\nBy difficulty level:")
print(f"  {'Level':>5s}  {'Acc':>5s}  {'k/n':>7s}")
print("  " + "-" * 22)
for lvl in sorted(lvl_stats.keys()):
    d = lvl_stats[lvl]
    acc = d["correct"] / d["total"] if d["total"] else 0
    print(f"  {str(lvl):>5s}  {acc:>5.0%}  {d['correct']:>3d}/{d['total']}")

Test accuracy : 55.3%
95% Wilson CI : [49.9%, 60.6%]
n = 329

By subject:
  Subject                           Acc      k/n
  ----------------------------------------------
  Algebra                           72%   81/113
  Intermediate Algebra              33%   28/85
  Number Theory                     52%   30/58
  Prealgebra                        59%   43/73

By difficulty level:
  Level    Acc      k/n
  ----------------------
      1    78%   25/32
      2    78%   46/59
      3    67%   42/63
      4    47%   40/85
      5    32%   29/90


- наблюдение один: простая алгебра решает довольно просто и хорошо , но уже к Intermediate Algebra  идет сильное снижение 72 до 33 процентов , как будто нужно более сложные инструменты 

- Capability cliff 

можно заметить что модель на первых уровнях показывала себя не плохо и уже на последнем уровне резкое подение до 32 , capability на приделах. какой то лимит на самых сложных заданиях 

- наблюдение 3:

такое ощущение что я немного переобучил модель , либо дал мало инструментов для выполнения бенча 

## Bonus assignment: Error Analysis

Pick 5-10 test samples that the agent got wrong and classify each failure. Here's one
possible taxonomy — feel free to use your own:

- **Tool misuse:** Called the wrong tool or with wrong arguments
- **Reasoning error:** Correct tool use but flawed multi-step reasoning
- **Format error:** Correct answer but wrong format in `submit()`
- **Inherent difficulty:** Problem requires reasoning beyond the tool set
- **Grading error:** The agent was actually right but the scorer got it wrong
- **Other:** Anything that doesn't fit the above

This kind of qualitative error analysis is essential in practice — aggregate metrics
tell you *how much* the agent struggles, but error analysis tells you *why*.

If you'd like to explore the logs in code, use the cell below — otherwise just read
them directly and delete it.

In [ ]:
# YOUR CODE HERE

---
1. What was the most common failure mode?
2. Which failure modes could be fixed with better prompting vs. better tools
   vs. a more capable model?
3. Did you find any grading errors? What does that imply about using model-graded scoring?

**Your answer:**